<h1>Estandarización y limpieza de Datos</h1>
En esta parte del proceso organizaremos nuestros datasets para que todos tengan la misma estructura

Primero importamos las librerias necesarias (pandas y os) y luego cargaremos nuestros datos

In [1]:
!pip install xlrd openpyxl pyarrow


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\Prueba\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import os

Crearemos un diccionario con nuestros datasets para que nos sea mas fácil iterar sobre ellos

In [3]:
# Carga tus datasets desde la carpeta raw
# Ajusta los nombres de archivo según los tuyos
# Carga datasets con engines y tipos correctos desde el origen
# Cargar archivos
df2020 = pd.read_excel("../data/raw/hurto_automotores_2020.xls", engine="xlrd", dtype={"CODIGO_DANE": str})
df2021 = pd.read_excel("../data/raw/hurto_automotores_2021.xlsx", engine="openpyxl", dtype={"CODIGO_DANE ": str})
df2022 = pd.read_excel("../data/raw/hurto_automotores_2022.xls", engine="xlrd", dtype={"CODIGO_DANE": str})
df2023 = pd.read_excel("../data/raw/hurto_automotores_2023.xlsx", engine="openpyxl", dtype={"CODIGO_DANE": str})
df2024 = pd.read_excel("../data/raw/hurto_automotores_2024.xlsx", engine="openpyxl", dtype={"CODIGO_DANE": str})
df2025 = pd.read_excel("../data/raw/hurto_automotores_2025.xlsx", engine="openpyxl", dtype={"CODIGO_DANE": str})

# Corrección de fechas seriales del 2021 ANTES de crear el diccionario
df2021["FECHA_HECHO"] = pd.to_datetime(df2021["FECHA_HECHO"], origin="1899-12-30", unit="D")

# Crear diccionario
datasets = {"dataset2020": df2020, "dataset2021": df2021, "dataset2022": df2022,
            "dataset2023": df2023, "dataset2024": df2024, "dataset2025": df2025}

<h2>Diagnostico</h2>

Usaremos el siguiente código para ver el orden y los tipos de datos de cada columna, haciendo un primer diagnostico de la estructura de nuestros datasets

In [4]:
# Ver columnas, orden y tipos de cada uno
for nombre, df in datasets.items():
    print(f"\n{'='*100}")
    print(f"📄 {nombre}")
    print(f"  Columnas ({len(df.columns)}): {df.columns.tolist()}")
    print(df.dtypes.to_string())
    print(df.sample(5))


📄 dataset2020
  Columnas (6): ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO_DANE', 'ARMAS_MEDIOS', 'FECHA_HECHO', 'CANTIDAD']
DEPARTAMENTO               str
MUNICIPIO                  str
CODIGO_DANE                str
ARMAS_MEDIOS               str
FECHA_HECHO     datetime64[us]
CANTIDAD                 int64
      DEPARTAMENTO MUNICIPIO CODIGO_DANE         ARMAS_MEDIOS FECHA_HECHO  \
4575  CUNDINAMARCA  GIRARDOT    25307000  SIN EMPLEO DE ARMAS  2020-02-14   
1661        TOLIMA     GUAMO    73319000        ARMA DE FUEGO  2020-02-21   
630      ATLÁNTICO   SOLEDAD    08758000        ARMA DE FUEGO  2020-02-09   
608      ATLÁNTICO    GALAPA    08296000        ARMA DE FUEGO  2020-11-21   
4562  CUNDINAMARCA     FUNZA    25286000  SIN EMPLEO DE ARMAS  2020-12-07   

      CANTIDAD  
4575         1  
1661         1  
630          1  
608          1  
4562         2  

📄 dataset2021
  Columnas (6): ['ARMAS_MEDIOS', 'DEPARTAMENTO', 'MUNICIPO', 'FECHA_HECHO', 'CODIGO_DANE', 'CANTIDAD ']
ARMAS_MEDIO

Este resultado nos permite ver que tenemos diferencias en el orden, el tipo y el nombre de nuestras columnas. Abordaremos estas diferencias. 

Usando nuestro primer dataset (año 2020) como referencia identificaremos las diferencias que tenemos en cuanto a los nombres de las columnas y el orden en el que estan. 

In [5]:
# Tomar el primer dataset como referencia
referencia = list(datasets.keys())[0]
cols_ref = datasets[referencia].columns.tolist()

print(f"Referencia: {referencia} → {cols_ref}\n")

for nombre, df in datasets.items():
    if nombre == referencia:
        continue
    cols = df.columns.tolist()
    
    # Columnas que faltan o sobran
    faltan  = set(cols_ref) - set(cols)
    sobran  = set(cols) - set(cols_ref)
    
    mismos_nombres = set(cols) == set(cols_ref)
    if mismos_nombres:
        orden_distinto = cols != cols_ref
    else:
        orden_distinto = "No aplica (hay columnas distintas)"
    
    print(f"🔍 {nombre}:")
    print(f"   Faltan  : {faltan  or 'ninguna'}")
    print(f"   Sobran  : {sobran  or 'ninguna'}")
    print(f"   Orden distinto: {orden_distinto}")

Referencia: dataset2020 → ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO_DANE', 'ARMAS_MEDIOS', 'FECHA_HECHO', 'CANTIDAD']

🔍 dataset2021:
   Faltan  : {'MUNICIPIO', 'CANTIDAD'}
   Sobran  : {'MUNICIPO', 'CANTIDAD '}
   Orden distinto: No aplica (hay columnas distintas)
🔍 dataset2022:
   Faltan  : {'DEPARTAMENTO', 'MUNICIPIO', 'CANTIDAD'}
   Sobran  : {'MUNICIPIO ', 'DEPARTAMENTO ', 'CANTIDAD '}
   Orden distinto: No aplica (hay columnas distintas)
🔍 dataset2023:
   Faltan  : {'DEPARTAMENTO', 'MUNICIPIO', 'CANTIDAD'}
   Sobran  : {'MUNICIPIO ', 'DEPARTAMENTO ', 'CANTIDAD '}
   Orden distinto: No aplica (hay columnas distintas)
🔍 dataset2024:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: True
🔍 dataset2025:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: True


Normalizaremos los nombres de la columna con el siguiente código

<h2>Estructura de las columnas</h2>

In [6]:
for nombre, df in datasets.items():
    df.columns = (df.columns
                  .str.strip()           # elimina espacios al inicio y al final
                  .str.lower()           # todo en minúsculas
                  .str.replace(" ", "_") # espacios del medio → guión bajo
                  )
    print(f"✅ {nombre}: {df.columns.tolist()}")

✅ dataset2020: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
✅ dataset2021: ['armas_medios', 'departamento', 'municipo', 'fecha_hecho', 'codigo_dane', 'cantidad']
✅ dataset2022: ['armas_medios', 'departamento', 'municipio', 'fecha_hecho', 'codigo_dane', 'cantidad']
✅ dataset2023: ['armas_medios', 'departamento', 'municipio', 'fecha_hecho', 'codigo_dane', 'cantidad']
✅ dataset2024: ['armas_medios', 'departamento', 'municipio', 'fecha_hecho', 'codigo_dane', 'cantidad']
✅ dataset2025: ['armas_medios', 'departamento', 'municipio', 'fecha_hecho', 'codigo_dane', 'cantidad']


corregimos el typo que aparece en el dataset 2021 (municipo)

In [7]:
df2021.rename(columns={"municipo": "municipio"}, inplace=True)

Corregimos el orden de acuerdo al orden de las columnas del primer dataset

In [8]:
orden_correcto = datasets["dataset2020"].columns.tolist()

for nombre, df in datasets.items():
    datasets[nombre] = df[orden_correcto]


Revisamos para ver si nuestras columnas quedaron iguales

In [9]:
for nombre, df in datasets.items():
    print(f"✅ {nombre}: {df.columns.tolist()}")

✅ dataset2020: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
✅ dataset2021: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
✅ dataset2022: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
✅ dataset2023: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
✅ dataset2024: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
✅ dataset2025: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']


<h2>Tipos de Datos
</h2>

Ahora normalizaremos los tipos de datos de cada columna de la siguiente manera: <br>  "departamento": "str",
    "municipio": "str",
    "codigo_dane": "str",
    "armas_medios": "str",
    "fecha_hecho": "datetime",
    "cantidad":  "Int64"

Finalmente convertiremos todas nuestras columnas al tipo de dato correcto

In [10]:
# Conversión especial de fechas del dataset 2021
datasets["dataset2021"]["fecha_hecho"] = pd.to_datetime(
    datasets["dataset2021"]["fecha_hecho"], dayfirst=True
)

In [11]:
for nombre, df in datasets.items():
    df["codigo_dane"] = df["codigo_dane"].astype(str)
    datasets[nombre] = df

In [12]:
tipos_correctos = {
    "departamento": "object",
    "municipio": "object",
    "codigo_dane": "object",
    "armas_medios": "object",
    "cantidad":  "Int64",
}

for nombre, df in datasets.items():
    # Primero los tipos normales (sin fecha)
    tipos_sin_fecha = {k: v for k, v in tipos_correctos.items() if v != "datetime64[ns]"}
    df = df.astype(tipos_sin_fecha)
    
    # Luego las fechas por separado
    df["fecha_hecho"] = pd.to_datetime(df["fecha_hecho"])
    datasets[nombre] = df
    print(f"✅ {nombre} fecha corregida")

for nombre, df in datasets.items():
    datasets[nombre] = df.astype(tipos_correctos)
    print(f"✅ {nombre} corregido")

✅ dataset2020 fecha corregida
✅ dataset2021 fecha corregida
✅ dataset2022 fecha corregida
✅ dataset2023 fecha corregida
✅ dataset2024 fecha corregida
✅ dataset2025 fecha corregida
✅ dataset2020 corregido
✅ dataset2021 corregido
✅ dataset2022 corregido
✅ dataset2023 corregido
✅ dataset2024 corregido
✅ dataset2025 corregido


C:\Users\Prueba\AppData\Local\Temp\ipykernel_14288\2469758419.py:15: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["fecha_hecho"] = pd.to_datetime(df["fecha_hecho"])


<h2>Nuevo Diagnóstico</h2>

In [13]:
# Ver columnas, orden y tipos de cada uno
for nombre, df in datasets.items():
    print(f"\n{'='*100}")
    print(f"📄 {nombre}")
    print(f"  Columnas ({len(df.columns)}): {df.columns.tolist()}")
    print(df.dtypes.to_string())


📄 dataset2020
  Columnas (6): ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
departamento            object
municipio               object
codigo_dane             object
armas_medios            object
fecha_hecho     datetime64[us]
cantidad                 Int64

📄 dataset2021
  Columnas (6): ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
departamento           object
municipio              object
codigo_dane            object
armas_medios           object
fecha_hecho     datetime64[s]
cantidad                Int64

📄 dataset2022
  Columnas (6): ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
departamento            object
municipio               object
codigo_dane             object
armas_medios            object
fecha_hecho     datetime64[us]
cantidad                 Int64

📄 dataset2023
  Columnas (6): ['departamento', 'municipio', 'codigo_dane', 'armas_medios',

Finalmente corremos de nuevo nuestros códigos de diagnostico y podemos ver que nuestras columnas y tipos de datos son iguales en todos los datasets


In [14]:
# Tomar el primer dataset como referencia
referencia = list(datasets.keys())[0]
cols_ref = datasets[referencia].columns.tolist()

print(f"Referencia: {referencia} → {cols_ref}\n")

for nombre, df in datasets.items():
    if nombre == referencia:
        continue
    cols = df.columns.tolist()
    
    # Columnas que faltan o sobran
    faltan  = set(cols_ref) - set(cols)
    sobran  = set(cols) - set(cols_ref)
    
    mismos_nombres = set(cols) == set(cols_ref)
    if mismos_nombres:
        orden_distinto = cols != cols_ref
    else:
        orden_distinto = "No aplica (hay columnas distintas)"
    
    print(f"🔍 {nombre}:")
    print(f"   Faltan  : {faltan  or 'ninguna'}")
    print(f"   Sobran  : {sobran  or 'ninguna'}")
    print(f"   Orden distinto: {orden_distinto}")

Referencia: dataset2020 → ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']

🔍 dataset2021:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: False
🔍 dataset2022:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: False
🔍 dataset2023:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: False
🔍 dataset2024:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: False
🔍 dataset2025:
   Faltan  : ninguna
   Sobran  : ninguna
   Orden distinto: False


<h2>LIMPIEZA DE DATOS </h2>

En esta etapa de nuestro analisis de datos vamos a estandarizar el texto de nuestras columnas eliminando espacios y poniendo todo en letra minúscula

In [15]:
for nombre, df in datasets.items():
    for col in df.select_dtypes(include="object").columns:
        df[col] = (df[col]
                   .str.strip()
                   .str.lower()
                   )
    datasets[nombre] = df
    print(f"✅ {nombre} texto estandarizado")

✅ dataset2020 texto estandarizado
✅ dataset2021 texto estandarizado
✅ dataset2022 texto estandarizado
✅ dataset2023 texto estandarizado
✅ dataset2024 texto estandarizado
✅ dataset2025 texto estandarizado


Luego revisamos los valores unicos de las columnas armas medios, departamento y municipio para estandarizar y unificar estos valores

In [16]:
for nombre, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"📄 {nombre}")
    print(f"\n  departamento: {sorted(df['departamento'].dropna().unique())}")
    print(f"\n  armas_medios: {sorted(df['armas_medios'].dropna().unique())}")


📄 dataset2020

  departamento: ['antioquia', 'arauca', 'atlántico', 'bolívar', 'boyacá', 'caldas', 'caquetá', 'casanare', 'cauca', 'cesar', 'chocó', 'cundinamarca', 'córdoba', 'guajira', 'huila', 'magdalena', 'meta', 'nariño', 'norte de santander', 'putumayo', 'quindío', 'risaralda', 'san andrés', 'santander', 'sucre', 'tolima', 'valle']

  armas_medios: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']

📄 dataset2021

  departamento: ['antioquia', 'arauca', 'atlántico', 'bolívar', 'boyacá', 'caldas', 'caquetá', 'casanare', 'cauca', 'cesar', 'chocó', 'cundinamarca', 'córdoba', 'guajira', 'huila', 'magdalena', 'meta', 'nariño', 'norte de santander', 'putumayo', 'quindío', 'risaralda', 'santander', 'sucre', 'tolima', 'valle']

  armas_medios: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'no resportado', 'sin empleo de armas']

📄 data

In [17]:
for nombre, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"📄 {nombre}")
    print(f"\n  municipios: {sorted(df['municipio'].dropna().unique())}")


📄 dataset2020

  municipios: ['abrego', 'acacías', 'aguachica', 'agustín codazzi', 'aipe', 'albania', 'albán', 'aldana', 'algarrobo', 'alvarado', 'amagá', 'anapoima', 'ancuyá', 'andalucía', 'andes', 'angelópolis', 'ansermanuevo', 'apartadó', 'apulo', 'aquitania', 'arauca (ct)', 'arauquita', 'arcabuco', 'argelia', 'ariguaní', 'armenia (ct)', 'armero', 'balboa', 'baranoa', 'barbosa', 'barrancabermeja', 'barrancas', 'barranquilla (ct)', 'becerril', 'bello', 'beltrán', 'bochalema', 'bogotá d.c. (ct)', 'bojacá', 'bosconia', 'bucaramanga (ct)', 'bucarasica', 'buenaventura', 'buenos aires', 'bugalagrande', 'cachipay', 'cajibío', 'cajicá', 'calamar', 'calarca', 'caldas', 'caldono', 'cali (ct)', 'calima', 'caloto', 'candelaria', 'caqueza', 'carepa', 'carmen de apicalá', 'cartagena (ct)', 'cartago', 'caucasia', 'chachagüí', 'chaparral', 'chimichagua', 'chinchiná', 'chipaque', 'chiriguaná', 'chocontá', 'chía', 'cimitarra', 'circasia', 'cisneros', 'ciudad bolívar', 'cogua', 'consaca', 'contadero'

Despues de analizar los valores unicos con la ayuda de Claude descubrimos que solo debemos arreglar errores en la columna armas medios, estandarizamos el texto de esta columna con el siguiente codigo

In [30]:
correcciones_armas = {
    'no resportado': 'no reportado',
    'no reporta':    'no reportado',
    '-':             'no reportado',
}

for nombre, df in datasets.items():
    datasets[nombre]["armas_medios"] = df["armas_medios"].replace(correcciones_armas)

# Verifica
for nombre, df in datasets.items():
    print(f"{nombre}: {sorted(df['armas_medios'].dropna().unique())}")

dataset2020: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']
dataset2021: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']
dataset2022: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']
dataset2023: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']
dataset2024: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']
dataset2025: ['arma blanca / cortopunzante', 'arma de fuego', 'contundentes', 'escopolamina', 'llave maestra', 'no reportado', 'sin empleo de armas']


Finalmente unimos nuestros datasets para hacer nuestro analisis sobre un solo archivo

In [19]:
df_total = pd.concat(datasets.values(), ignore_index=True)

# Verifica
print(f"Filas totales: {len(df_total)}")
print(f"Columnas: {df_total.columns.tolist()}")
print(df_total.sample(5))

Filas totales: 43076
Columnas: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']
       departamento         municipio codigo_dane  \
22896  cundinamarca  bogotá d.c. (ct)    11001000   
13037         cauca            timbío    19807000   
5580      antioquia          sabaneta     5631000   
24063     antioquia     medellín (ct)    05001000   
36376  cundinamarca  bogotá d.c. (ct)    11001000   

                      armas_medios fecha_hecho  cantidad  
22896                llave maestra  2023-10-02         1  
13037  arma blanca / cortopunzante  2022-06-13         1  
5580                 arma de fuego  2021-10-28         1  
24063          sin empleo de armas  2023-01-12         1  
36376                arma de fuego  2025-11-03         1  


Sobre nuestro dataset total revisamos valores nulos y duplicados

In [20]:
# Cuántos nulos hay en cada columna
print(df_total.isnull().sum())

departamento    0
municipio       0
codigo_dane     0
armas_medios    0
fecha_hecho     0
cantidad        0
dtype: int64


In [21]:
# Cuántos filas completamente duplicadas hay
print(f"Filas duplicadas: {df_total.duplicated().sum()}")

Filas duplicadas: 4712


In [22]:
# Ver los duplicados
duplicados = df_total[df_total.duplicated(keep=False)]
print(f"Total filas involucradas: {len(duplicados)}")
print(duplicados.sort_values(by=["fecha_hecho", "municipio"]).head(20))

Total filas involucradas: 7270
       departamento         municipio codigo_dane  \
3266   cundinamarca  bogotá d.c. (ct)    11001000   
5265   cundinamarca  bogotá d.c. (ct)    11001000   
3276   cundinamarca  bogotá d.c. (ct)    11001000   
5267   cundinamarca  bogotá d.c. (ct)    11001000   
7474      antioquia     medellín (ct)     5001000   
11065     antioquia     medellín (ct)     5001000   
6688   cundinamarca  bogotá d.c. (ct)    11001000   
7974   cundinamarca  bogotá d.c. (ct)    11001000   
10582  cundinamarca  bogotá d.c. (ct)    11001000   
11339  cundinamarca  bogotá d.c. (ct)    11001000   
5310   cundinamarca  bogotá d.c. (ct)    11001000   
6689   cundinamarca  bogotá d.c. (ct)    11001000   
10583  cundinamarca  bogotá d.c. (ct)    11001000   
12428  cundinamarca  bogotá d.c. (ct)    11001000   
8714          valle         cali (ct)    76001000   
11773         valle         cali (ct)    76001000   
7475      antioquia     medellín (ct)     5001000   
9519      antio

In [23]:
# Agregar columna temporal con el año para identificar origen
for nombre, df in datasets.items():
    df["año_origen"] = nombre

df_total_debug = pd.concat(datasets.values(), ignore_index=True)
duplicados = df_total_debug[df_total_debug.duplicated(keep=False)]
print(duplicados["año_origen"].value_counts())

año_origen
dataset2025    5273
dataset2021    1371
dataset2023     622
dataset2020       4
Name: count, dtype: int64


In [24]:
dup_2025 = datasets["dataset2025"][datasets["dataset2025"].duplicated(keep=False)]
print(dup_2025.groupby("municipio").size().sort_values(ascending=False).head(15))

municipio
bogotá d.c. (ct)          2948
cali (ct)                 1089
medellín (ct)              629
barranquilla (ct)          164
popayán (ct)                54
bello                       42
soacha                      37
ipiales                     20
santander de quilichao      20
itagui                      16
cartagena (ct)              12
palmira                     11
cúcuta (ct)                 11
pasto (ct)                  10
jamundí                     10
dtype: int64


In [25]:
bogota_2025 = datasets["dataset2025"][datasets["dataset2025"]["municipio"] == "bogotá d.c. (ct)"]
print(f"Total hurtos Bogotá 2025: {len(bogota_2025)}")
print(f"\nHurtos por día (primeros 10 días con más hurtos):")
print(bogota_2025.groupby("fecha_hecho").size().sort_values(ascending=False).head(10))

Total hurtos Bogotá 2025: 3246

Hurtos por día (primeros 10 días con más hurtos):
fecha_hecho
2025-11-13    25
2025-10-18    22
2025-10-19    19
2025-06-14    19
2025-02-15    18
2025-11-11    18
2025-04-12    18
2025-11-08    18
2025-02-08    17
2025-09-24    17
dtype: int64


In [26]:
bogota_dup = bogota_2025[bogota_2025.duplicated(keep=False)]
print(f"\nFilas involucradas en duplicados Bogotá: {len(bogota_dup)}")
print(f"\nDuplicados por arma:")
print(bogota_dup.groupby("armas_medios").size().sort_values(ascending=False))


Filas involucradas en duplicados Bogotá: 2948

Duplicados por arma:
armas_medios
llave maestra                  1573
arma de fuego                   764
sin empleo de armas             597
arma blanca / cortopunzante      10
escopolamina                      4
dtype: int64


In [27]:
bogota_2025 = datasets["dataset2025"][datasets["dataset2025"]["municipio"] == "bogotá d.c. (ct)"]

# Promedio de hurtos por día en Bogotá
total_dias = bogota_2025["fecha_hecho"].nunique()
total_hurtos = len(bogota_2025)
promedio_dia = total_hurtos / total_dias

print(f"Total hurtos Bogotá 2025: {total_hurtos}")
print(f"Días con registros: {total_dias}")
print(f"Promedio hurtos por día: {round(promedio_dia, 2)}")

# Cuántos días aparece cada arma como principal
print(f"\nDías únicos por arma:")
print(bogota_2025.groupby("armas_medios")["fecha_hecho"].nunique())

# Promedio de hurtos por día por arma
print(f"\nPromedio hurtos por día por arma:")
print(bogota_2025.groupby("armas_medios").apply(
    lambda x: round(len(x) / x["fecha_hecho"].nunique(), 2)
))

Total hurtos Bogotá 2025: 3246
Días con registros: 364
Promedio hurtos por día: 8.92

Días únicos por arma:
armas_medios
arma blanca / cortopunzante     63
arma de fuego                  317
contundentes                     9
escopolamina                    19
llave maestra                  354
no reportado                     8
sin empleo de armas            286
Name: fecha_hecho, dtype: int64

Promedio hurtos por día por arma:
armas_medios
arma blanca / cortopunzante    1.08
arma de fuego                  2.64
contundentes                   1.00
escopolamina                   1.11
llave maestra                  4.53
no reportado                   1.00
sin empleo de armas            2.44
dtype: float64


 NOTA METODOLÓGICA
 El dataset contiene filas con valores idénticos en todas las columnas.
 Tras análisis estadístico se determinó que estos registros corresponden
 a hechos reales distintos ocurridos el mismo día en el mismo municipio
 con las mismas características, lo cual es estadísticamente plausible
 dado el volumen de hurtos en ciudades principales como Bogotá (promedio
 8.92 hurtos/día) y la granularidad limitada de las variables disponibles.
 No se eliminaron filas por ausencia de identificador único por hecho.

Agregamos columna de id a nuestro dataset total

In [28]:
# Agregar columna identificadora única
df_total = df_total.reset_index(drop=True)
df_total.insert(0, "id_registro", df_total.index + 1)

<strong>Exportamos nuestro dataset total</strong>

In [29]:
# Exportar dataset consolidado y limpio
os.makedirs("../data/processed", exist_ok=True)
df_total.to_parquet("../data/processed/hurto_automotores_consolidado.parquet", index=False)
print("✅ Dataset exportado correctamente")

# Verificación final
df_verificacion = pd.read_parquet("../data/processed/hurto_automotores_consolidado.parquet")
print(f"\nFilas: {len(df_verificacion)}")
print(f"Columnas: {df_verificacion.columns.tolist()}")
print(f"\nTipos de datos:")
print(df_verificacion.dtypes.to_string())

✅ Dataset exportado correctamente

Filas: 43076
Columnas: ['id_registro', 'departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'cantidad']

Tipos de datos:
id_registro              int64
departamento               str
municipio                  str
codigo_dane                str
armas_medios               str
fecha_hecho     datetime64[us]
cantidad                 Int64
